In [1]:
#importing packages

import cv2
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import os
import pandas as pd
from pathlib import Path


directory = Path('C:/work/characterization_acquisition_time/Paco - IGR/margin') # TO CHANGEE: directory with the samples
samples = [folder.name for folder in directory.iterdir() if folder.is_dir()] # Extracting the subdirectories corresponding to each sample



data = {} # A dictionary to store results for each sample
entropy_bin = 5 # Bin in angles for entropy calculation, as explained in the paper

# Function to calculate the circular variance, given doubled angles (originally between 1 and 179) in radians
# The circular variance is calculated as described in the paper
def circular_variance(angles):
    n = len(angles)
    sum_cos = np.sum(np.cos(angles))
    sum_sin = np.sum(np.sin(angles))
    R = np.sqrt(sum_cos**2 + sum_sin**2) / n
    variance = 1 - R
    return variance

# Looping over all samples to calculate the entropy and circular vairance over all filtered pixels (R2>0.7) 
for sam in samples: 
    data[sam] =[]
    image = tifffile.imread(os.path.join(directory, sam, "ch03_PHI_angR2filtH_1S_R2V_stitched_noLB.tif")) # extracting the p-SHG orientation map as saved in the mosaicking program
    image_r2 = tifffile.imread(os.path.join(directory, sam, "ch03_R2_filtered_stitched_noLB.tif")) # extracting the R2 map to filter the data
    hsv_array = cv2.cvtColor(image, cv2.COLOR_RGB2HSV) # converting the RGB in HSVto extract orientation
    h_channel, s_channel, v_channel = cv2.split(hsv_array) # h chqnnel: orientation and v channel: R2
    h_channel = h_channel.astype(float)  # Converting to float to allow NaN
    h_channel[image_r2/10000 < 0.7] =np.nan # filtering by putting NaN for all pixels with R2<0.7. Note R2 zqs sqved *10000 in 16bits for accurate values storage
    mask = image_r2 / 10000 >= 0.7
    percentage = 100*np.sum(mask)/mask.size # calculating the percentage of filtered pixels in the image

    # converting all filtered orientation values in a flattened list and removal of Nan
    flat_arr = h_channel.flatten() 
    flat_arr =  flat_arr[~np.isnan(flat_arr)]
    flat_arr = flat_arr.astype(int)

    # circular variance calculation on the doubled orientation values in radians
    variance = circular_variance(2*flat_arr*np.pi/180)

    # entropy calculation as explained in the paper using bins as precised previously
    bins = np.arange(0, 181, entropy_bin)
    hist, _ = np.histogram(flat_arr, bins=bins)
    probabilities = hist / len(flat_arr)
    probabilities = probabilities[probabilities > 0]
    entropy = -np.sum(probabilities * np.log(probabilities))/np.log(len(bins)-1)

    # sotring the values
    data[sam].append(round(variance,3))
    data[sam].append(round(entropy,3))
    data[sam].append(round(percentage,2))

#saving values in excel 
results_df = pd.DataFrame(data)
results_df = results_df.T
results_df.to_excel(os.path.join(directory,'summary_margin_Orient07_pSHG07.xlsx'), index=True) 

NameError: name 'samples' is not defined